# Train a GPT with BPE Tokenizer on WikiText-2

This notebook trains a GPT with subword tokenization (BPE) on WikiText-2:

```
wikitext2.txt (~2.5M tokens, Wikipedia articles)
    → [BPETokenizer.train()] → learn 512 subword vocabulary
    → [BPETokenizer.encode()] → shorter sequences
    → [CharLevelDataset]
    → [GPT with dense FFN or MoEFFN] → logits
    → [CrossEntropyLoss + aux_loss] → loss
    → [AdamW optimizer] → gradient update
    → [repeat] → trained model → generate text!
```

Set `USE_MOE = True` below to replace the dense FeedForward in each
block with a sparse Mixture of Experts (8 experts, top-2 routing).

### Why WikiText-2?

TinyShakespeare (~200K tokens) is too small for MoE — the extra expert
parameters just memorise the data. WikiText-2 (~2.5M tokens) gives
enough data for the sparse MoE to generalise better than a dense model
at the same active-parameter count.

### Key Concept: Label Shift

For a sequence of tokens $x_1, x_2, \dots, x_T$, the model predicts
$\hat{x}_2, \hat{x}_3, \dots, \hat{x}_{T+1}$ — the **next token** at each position.

In [ ]:
import sys
from pathlib import Path

# Add project root so we can import core.*
project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import math

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from core.transformer import (
    GPT,
    BPETokenizer,
    create_dataloaders,
)
from core.transformer.data import load_corpus_from_file
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────
# BPE vocabulary
BPE_VOCAB_SIZE = 2048  # Target subword vocabulary size

# Model dimensions (Dense baseline for MoE fair comparison)
D_MODEL = 256  # Embedding / transformer dimension
N_HEADS = 8  # Number of attention heads
N_KV_HEADS = 4  # GQA: KV heads (None = same as N_HEADS, i.e. standard MHA)
N_LAYERS = 4  # Number of GPTBlocks
D_FF = 768  # FeedForward hidden dim (3 * D_MODEL)

# MoE (set USE_MOE=True to replace dense FFN with sparse Mixture of Experts)
USE_MOE = True
MOE_N_EXPERTS = 8
MOE_K = 2

# Training
BLOCK_SIZE = 256  # Context length (T)
BATCH_SIZE = 64  # Sequences per batch
MAX_STEPS = 10000  # Total training steps
LR = 3e-4  # Peak learning rate
DROPOUT = 0.1  # Dropout rate
WEIGHT_DECAY = 0.1  # AdamW weight decay
WARMUP_STEPS = 500  # Linear warmup steps
GRAD_CLIP = 1.0  # Max gradient norm
EVAL_INTERVAL = 500  # Steps between evaluations
GEN_INTERVAL = 500  # Steps between text generation

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

---
## 1. Train BPE Tokenizer & Load Data

First, train a BPE tokenizer on TinyShakespeare to learn 512 subword tokens,
then load the data with the subword tokenizer.

In [ ]:
# ── Load corpus from local file ──────────────────────────────────
corpus = load_corpus_from_file(Path(project_root) / "assets" / "wikitext2.txt")

# ── Load or train BPE tokenizer ──────────────────────────────────
SAVE_DIR = Path(project_root) / "models"
tokenizer_path = SAVE_DIR / "bpe_tokenizer.pt"

if tokenizer_path.exists():
    tokenizer_data = torch.load(tokenizer_path, map_location="cpu", weights_only=True)
    tokenizer = BPETokenizer(
        vocab_size=len(tokenizer_data["vocab"]),
        special_tokens=tokenizer_data["special_tokens"],
        regex_pattern=tokenizer_data["regex_pattern"],
    )
    tokenizer.vocab = tokenizer_data["vocab"]
    tokenizer.id_to_token = tokenizer_data["id_to_token"]
    tokenizer.merges = tokenizer_data["merges"]
    tokenizer.merge_ranks = tokenizer_data["merge_ranks"]
    print(f"Loaded BPE tokenizer from {tokenizer_path} ({tokenizer.vocab_size} vocab)")
else:
    print("No pre-trained tokenizer found — training from scratch...")
    tokenizer = BPETokenizer(vocab_size=BPE_VOCAB_SIZE)
    tokenizer.train([corpus])

VOCAB_SIZE = tokenizer.vocab_size
print(f"BPE tokenizer: target={BPE_VOCAB_SIZE}, actual={VOCAB_SIZE}")
print(f"  Learned merges: {len(tokenizer.merges)}")
print(
    f"  Base chars:     {VOCAB_SIZE - len(tokenizer.merges) - len(tokenizer.special_tokens)}"
)
print()

# ── Show compression ─────────────────────────────────────────────
raw_len = len(corpus)
bpe_len = len(tokenizer.encode(corpus))
print(f"Compression: {raw_len:,} chars → {bpe_len:,} tokens ({raw_len / bpe_len:.2f}x)")
print()

# ── Subword examples ──────────────────────────────────────────────
merged = sorted(
    [(v, k) for k, v in tokenizer.vocab.items() if len(k) > 1 and not k.startswith("<")]
)
print(f"Sample subwords: {[t for _, t in merged[:15]]}")
print()

# ── Encode example ────────────────────────────────────────────────
example = "An important goal of AI research is to allow computers to communicate in natural languages like English . "
ids = tokenizer.encode(example)
tokens = [tokenizer.id_to_token[i] for i in ids]
print(f'Encode example: "{example}"')
print(f"  → {tokens}")
print(
    f"  → {len(example)} chars → {len(ids)} tokens  (compression={len(example) / len(ids):.1f}x)"
)
print()

# ── Create DataLoaders ────────────────────────────────────────────
train_loader, val_loader, train_ds, val_ds = create_dataloaders(
    corpus_name="wikitext2",
    tokenizer=tokenizer,
    block_size=BLOCK_SIZE,
    batch_size=BATCH_SIZE,
    data_dir=str(Path(project_root) / "assets"),
)

print(f"Corpus encoded length: {len(train_ds.data) + len(val_ds.data)} tokens")
print(f"Train samples: {len(train_ds)}  (= {len(train_loader)} batches/epoch)")
print(f"Val samples:   {len(val_ds)}  (= {len(val_loader)} batches)")
print()

x_batch, y_batch = next(iter(train_loader))
print(f"Batch x shape: {x_batch.shape}  (batch_size, seq_len)")
print(f"Batch y shape: {y_batch.shape}  (batch_size, seq_len)")

---
## 2. Model Initialisation

Create a mini GPT with the hyperparameters above. When `USE_MOE=True`,
the dense FeedForward in each block is replaced with a sparse Mixture
of Experts (MoEFFN) using 8 experts, top-2 routing — more total
parameters, similar active FLOPs per token.

In [ ]:
model = GPT(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    max_seq_len=BLOCK_SIZE,
    d_ff=D_FF,
    dropout=DROPOUT,
    use_moe=USE_MOE,
    n_experts=MOE_N_EXPERTS,
    moe_k=MOE_K,
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())

active_per_token_ffn = 2 * D_MODEL * D_FF + D_FF + D_MODEL  # one FFN: in+out
if USE_MOE:
    active_per_token_ffn *= MOE_K  # only k experts active per token

kv_heads = N_KV_HEADS if N_KV_HEADS is not None else N_HEADS
kv_saving = f" ({kv_heads}/{N_HEADS} KV heads)" if kv_heads < N_HEADS else ""

print(model)
print(f"Total params: {total_params:,}")
if USE_MOE:
    print(f"  MoE: {MOE_N_EXPERTS} experts, k={MOE_K}")
print(f"  Active FFN params per token: ~{active_per_token_ffn:,}")
print(f"  KV cache: {kv_heads} heads{kv_saving}")

---
## 3. Optimizer & LR Schedule

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def lr_lambda(step: int) -> float:
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

---
## 4. Training Loop

This is the core of the notebook. Each training step:

1. Get a batch `(x, y)` from the train DataLoader
2. Forward pass: `logits = model(x)`
3. Compute loss: `F.cross_entropy(logits.view(-1, V), y.view(-1))`
4. **If MoE**: add auxiliary load-balancing loss: `loss = nll + 1e-2 * aux`
5. Backward pass: `loss.backward()`
6. Gradient clipping: `torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)`
7. Update: `optimizer.step()` + `scheduler.step()`
8. Log: print step, losses, perplexity, LR
9. Every EVAL_INTERVAL: compute validation loss
10. Every GEN_INTERVAL: generate a sample to see the model improving

In [ ]:
train_losses = []
val_losses = []
step_lrs = []
step = 0

model.train()

while step < MAX_STEPS:
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        logits = model(x)

        # ── NLL loss (next-token prediction) ──────────────────────────
        nll = F.cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            y.view(-1),
            label_smoothing=0.1,
        )

        # ── MoE auxiliary load-balancing loss ─────────────────────────
        if USE_MOE:
            aux_loss = model.aux_loss
            loss = nll + 1e-2 * aux_loss
        else:
            aux_loss = 0.0
            loss = nll

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()
        scheduler.step()

        train_losses.append(loss.item())
        step_lrs.append(scheduler.get_last_lr()[0])

        if step % 100 == 0:
            ppl = math.exp(nll.item())
            msg = f"Step {step:5d} | loss {loss.item():.4f} | nll {nll.item():.4f}"
            if USE_MOE:
                msg += f" | aux {aux_loss.item():.4e}"
            msg += f" | ppl {ppl:.2f} | lr {step_lrs[-1]:.2e}"
            print(msg)

        if step > 0 and step % EVAL_INTERVAL == 0:
            model.eval()
            val_loss = 0.0
            n = 0
            with torch.no_grad():
                for i, (xv, yv) in enumerate(val_loader):
                    if i >= 500:
                        break
                    xv, yv = xv.to(DEVICE), yv.to(DEVICE)
                    logits = model(xv)
                    vloss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yv.view(-1))
                    val_loss += vloss.item() * len(xv)
                    n += len(xv)
            val_loss /= n
            val_losses.append(val_loss)
            print(
                f"  └─ Val loss {val_loss:.4f} | val ppl {math.exp(val_loss):.2f} (n={n})"
            )
            model.train()

        if step > 0 and step % GEN_INTERVAL == 0:
            model.eval()
            prompt_ids = torch.tensor(
                [tokenizer.encode("An important goal of AI research is to ")],
                dtype=torch.long,
                device=DEVICE,
            )
            with torch.no_grad():
                output = model.generate(prompt_ids, max_new_tokens=80, temperature=0.8)
            sample_text = tokenizer.decode(output[0].tolist())
            print(f"  └─ Sample [{step}]: {sample_text[:120]}...")
            model.train()

        step += 1
        if step >= MAX_STEPS:
            break

print("Training complete!")

---
## 5. Training Curves

Plot the loss and perplexity over time to verify learning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

model_type = "MoE" if USE_MOE else "Dense"

# Training loss curve
ax = axes[0]
ax.plot(train_losses, alpha=0.6, label=f"{model_type} train loss")
if val_losses:
    val_steps = list(range(0, len(train_losses), EVAL_INTERVAL))[: len(val_losses)]
    ax.plot(val_steps, val_losses, "o-", label=f"{model_type} val loss", markersize=4)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title(f"Training & Validation Loss ({model_type})")
ax.legend()
ax.grid(True, alpha=0.3)

# Learning rate schedule
ax = axes[1]
ax.plot(step_lrs)
ax.set_xlabel("Step")
ax.set_ylabel("Learning Rate")
ax.set_title("LR Schedule (Warmup + Cosine)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Text Generation

Now that the model is trained, let's see what it learned!

Generate from the same prompt with different temperatures to see how
the output quality changes.

In [ ]:
# ── Compare sampling strategies ────────────────────────────────────────
model.eval()
prompt = "An important goal of AI research is to "
prompt_ids = tokenizer.encode(prompt)
prompt_tensor = torch.tensor([prompt_ids], dtype=torch.long).to(DEVICE)

temperatures = [0.0, 0.5, 0.8, 1.2]
names = ["Greedy (T=0.0)", "Conservative (T=0.5)", "Creative (T=0.8)", "Random (T=1.2)"]

for temp, name in zip(temperatures, names):
    with torch.no_grad():
        output_ids = model.generate(
            prompt_tensor,
            max_new_tokens=200,
            temperature=temp,
            top_k=40,
            top_p=0.9,
        )
    generated = tokenizer.decode(output_ids[0].tolist())
    print(f"═══ {name} ═══")
    print(generated)
    print()

---
## 7. Save Model Weights

Save the trained model so we can load it in analysis notebooks
without retraining from scratch. Also save the tokenizer vocab
so the analysis notebook knows the character mappings.

In [ ]:
save_dir = Path(project_root) / "models"
save_dir.mkdir(parents=True, exist_ok=True)

# Save model state_dict
model_path = save_dir / "gpt_tinyshakespeare.pt"
torch.save(model.state_dict(), model_path)
print(
    f"Model weights saved to {model_path} ({model_path.stat().st_size / 1024:.1f} KB)"
)

# Save BPE tokenizer (vocab + merges + config)
tokenizer_path = save_dir / "bpe_tokenizer.pt"
torch.save(
    {
        "vocab": tokenizer.vocab,
        "id_to_token": tokenizer.id_to_token,
        "merges": tokenizer.merges,
        "merge_ranks": tokenizer.merge_ranks,
        "special_tokens": tokenizer.special_tokens,
        "regex_pattern": tokenizer.regex_pattern,
    },
    tokenizer_path,
)
print(f"BPE tokenizer saved to {tokenizer_path}")

config = {
    "d_model": D_MODEL,
    "n_layers": N_LAYERS,
    "n_heads": N_HEADS,
    "n_kv_heads": N_KV_HEADS if N_KV_HEADS is not None else N_HEADS,
    "d_ff": D_FF,
    "block_size": BLOCK_SIZE,
    "vocab_size": VOCAB_SIZE,
    "max_steps": MAX_STEPS,
    "final_train_loss": train_losses[-1] if train_losses else None,
    # Phase 6: MoE parameters
    "use_moe": USE_MOE,
    "n_experts": MOE_N_EXPERTS,
    "moe_k": MOE_K,
}
torch.save(config, save_dir / "training_config.pt")
print("Training config saved")